# Read and write one small metadata file

Use the checked input/output template to save a record, load it again and confirm
that an existing file is protected. This is a documentation example format, not
an EPSM building artifact or the future platform API.

**Before starting:** Python 3.11 or newer; no extra packages. Start in this
repository or `examples/`. The fixture source ID is
`synthetic.handbook.input-output.v1`. Counts are dimensionless. No random sampling
or seed is used. Expected outcome: a matching round trip and a refused accidental
overwrite, using an automatically cleaned temporary directory.


## Load the checked input/output implementation

Read the repository template explicitly with `runpy`. The notebook imports its
functions and does not copy their implementation. Missing template files must
fail clearly.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import runpy

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent

template_path = repo_root / "docs/templates/io-module.py.tmpl"
if not template_path.is_file():
    raise FileNotFoundError("Open this notebook from the repository root or examples folder")

template = runpy.run_path(str(template_path))
read_metadata = template["read_metadata"]
write_metadata = template["write_metadata"]


## Prepare a small, explicit input

The example file has exactly three fields: a format version, identifier and loaded
item count. The format version identifies this example JSON layout; it is not a
catalogue or database release.


In [ ]:
fixture = {"format_version": 1, "identifier": "offline-example", "item_count": 2}
fixture


## Write, read and protect the existing file

`TemporaryDirectory` creates an existing output directory and removes it after
the block. The writer stages a complete file before publishing it. By default it
refuses to overwrite an existing path. The reader returns the validated metadata.


In [ ]:
with TemporaryDirectory(prefix="epsm-handbook-") as directory:
    output_path = Path(directory) / "metadata.json"
    written_path = write_metadata(fixture, output_path)
    restored = read_metadata(written_path)
    assert written_path == output_path
    assert restored == fixture

    try:
        write_metadata(fixture, output_path)
    except FileExistsError:
        pass
    else:
        raise AssertionError("The default writer must refuse an existing output")

    assert read_metadata(output_path) == fixture
    print(restored)


## Check the result and cleanup

The loaded result remains available after the temporary file is removed. No
repository file was written or overwritten. Retained study outputs would instead
use an explicitly chosen caller-owned directory.


In [ ]:
assert restored == fixture
assert not output_path.exists()
restored


## Check a fresh run

Use **Restart Kernel and Run All**. Every assertion must pass, including overwrite
protection and cleanup. Clear outputs and execution counts before committing.
This check demonstrates the example file format; it does not validate an EPSM
package handoff, engine run or scientific calculation.
